# bn-weight-bias-init-pattern — worked example 1: Apply DCGAN BN init and count touched modules

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bn-weight-bias-init-pattern`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

DCGAN initializes every BatchNorm layer's multiplicative weight (gamma) from `N(1, 0.02)` and sets its additive bias (beta) to exactly zero. The `model.apply(fn)` call walks every submodule recursively, so `fn` must itself test `isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))` and skip everything else. Non-BN layers (Conv, Linear) are left untouched.

## Worked solution

**Step 1 — write a per-module `init_fn`.** `model.apply` visits *every* submodule, so the function receives Conv, Linear, BN, and even the container module itself. We guard with `isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))` so only BN layers are mutated. This is *why* we cannot blindly call `nn.init.normal_(m.weight, ...)` — most modules have a `weight` we must NOT touch.

**Step 2 — set gamma and beta.** Inside the guard, `nn.init.normal_(m.weight, 1.0, 0.02)` resamples gamma centered at 1 (BN *multiplies* by gamma, so 0 would zero the signal — 1 is the identity scale). `nn.init.zeros_(m.bias)` makes beta exactly 0, the identity shift.

**Step 3 — count via a side-effect closure.** To prove the guard works, we keep a list `touched` and append each BN module's name-free identity. After `model.apply(init_fn)`, `len(touched)` equals the number of BN layers only — the two Conv and one Linear are skipped.

**Step 4 — verify.** We check every BN bias is all-zero and that the touched count matches the number of BN submodules counted independently. The Linear weight is unchanged from its default, confirming non-BN layers were left alone.

In [ ]:
import torch.nn as nn

t.manual_seed(0)

model = nn.Sequential(
    nn.Conv2d(3, 8, 3),
    nn.BatchNorm2d(8),
    nn.Conv2d(8, 16, 3),
    nn.BatchNorm2d(16),
    nn.Flatten(),
    nn.Linear(16, 4),
    nn.BatchNorm1d(4),
)

touched = []

def init_fn(m):
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)
        touched.append(m)

model.apply(init_fn)

n_bn = sum(isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)) for m in model.modules())
all_bias_zero = all(t.allclose(m.bias, t.zeros_like(m.bias)) for m in touched)
print("BN layers touched:", len(touched), "of", n_bn, "BN submodules")
print("all BN biases zero:", all_bias_zero)
print("first BN gamma mean (~1.0):", round(touched[0].weight.mean().item(), 3))